# Single Baseline 2D Wiener Inpainting

**by Josh Dillon and Tyler Cox**, last updated October 14, 2026

This notebook inpaints the corner-turned, calibrated, redundantly-averaged single-baseline data under the night's final flags:
the last step of the per-night pipeline, whose products go to LST stacking. Every flagged cell, autocorrelations included, is
filled by a 2D Wiener fill about a *prior*: the conditional mean of a Gaussian process regression (GPR) of the
residual between the data and the prior, with a covariance that is stationary in time and frequency and learned from the night's
own residual. It replaces the feathered 1D DPSS inpainting of H6C's
[single_baseline_scaffolded_and_feathered_inpainter](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/single_baseline_scaffolded_and_feathered_inpainter.ipynb)
and
[single_baseline_2D_informed_inpaint](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/single_baseline_2D_informed_inpaint.ipynb).

The prior:
* **With a sky model:** the LST-indexed single-baseline sky model of the same redundant group, matched to the night's LSTs and
  rephased, then fit to the night by a smooth gain correction and an additive correction out to the horizon (a ridge pulls the
  modes the data constrain poorly back to no adjustment).
* **Without one, where it is flagged, and always for autocorrelations** (a multi-night stack's autos fit one night's poorly): a 2D DPSS
  fit of the night's own data over the sky's fringe rates, out to the horizon plus `DATA_PRIOR_DELAY_BUFFER` in delay, and at least
  `DATA_PRIOR_MIN_DELAY` for cross-correlations, so that it carries the mutual coupling. For a cross-correlation, ridges pull the
  combinations of modes that a wide gap leaves unconstrained back to zero, so that the fit solves a well-posed problem rather than
  depending on where its solver stops:
    * a light one (`DATA_PRIOR_RIDGE`) on the modes out to the horizon plus the buffer;
    * a heavier one (`DATA_PRIOR_EXTENSION_RIDGE`) on the modes past it, so that they do not pop up.

  An autocorrelation is bright enough that its plain fit needs no ridge.
* **Where a model covers only part of the night** (an LST range the stack lacks): a first fill about that mixed prior is refit by the
  data-only fit, and the model is blended into the refit over the last `WIENER_TAPER_HOURS` before each edge of its coverage, so
  that the prior does not step there.

The fill works in autocorrelation units (the residual over $\sqrt{|A_i A_j|}$, from a smooth 2D DPSS model of the night's
autocorrelations), with the noise predicted from the autos, the effective `nsamples` and the correlator's design (`CORRELATOR`). Its covariance is the residual's 2D
power spectrum in fringe rate and delay:
* from masked lag products, with Blackman-Harris lag tapers of equivalent widths `WIENER_TAPER_HOURS` and `WIENER_TAPER_MHZ`;
* with the noise model's flat floor subtracted;
* keeping only the modes above `WIENER_THRESHOLD_SIGMA` times the estimate's scatter;
* learned, where a model covers only part of the night, from the covered integrations if they span at least twice the taper's reach.

The fill is solved by conjugate gradients with FFT convolutions, and each polarization and band's covariance is written out
(`sum.inpaint_covariance.h5`) for assessing the noise of the inpainted and LST-stacked data.

The polarized point source models are subtracted first, so the inpainted data stay source-subtracted. At inpainted cells, the
effective `nsamples` are filled in by a low-delay DPSS fit across the gap, so that LST stacking weights the inpainted data by
the depth the night would have had there; the `where_inpainted` sidecar records which cells those are.

Configuration comes from a TOML file (e.g.
`hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml`) pointed to by the
`TOML_FILE` environment variable; if none is given, the default settings in the cells below are
used. Environment variables otherwise carry only paths and wrapper-level toggles.

Here's a set of links to skip to particular figures and tables:

• [Figure 1: Adjustment of the Sky Model to the Night](#Figure-1:-Adjustment-of-the-Sky-Model-to-the-Night)

• [Figure 2: Covariance Learned by the 2D Wiener Fill](#Figure-2:-Covariance-Learned-by-the-2D-Wiener-Fill)

• [Figure 3: 4-Pol Phase and Amplitude Waterfalls Before and After Inpainting](#Figure-3:-4-Pol-Phase-and-Amplitude-Waterfalls-Before-and-After-Inpainting)

In [ ]:
import time
tstart = time.time()
!hostname
!date

In [ ]:
import os
import toml
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import h5py
import hdf5plugin  # REQUIRED to have the compression plugins available
import numpy as np
import yaml
import glob
import copy
import re
from astropy import units
from scipy import constants
import scipy.fft as sfft
import scipy.sparse.linalg as spla

from pyuvdata import UVFlag
from hera_cal import io, flag_utils, utils, polfilt, abscal, red_groups, datacontainer
from hera_cal.noise import correlator_noise_covariance
from hera_cal.frf import sky_frates, get_FR_buffer_from_spectra
from hera_filters.dspec import dpss_operator, sparse_linear_fit_2D, fourier_filter, gen_window
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display
%matplotlib inline

## Parse inputs and outputs

To run interactively, set `SUM_FILE` (and `TOML_FILE`, if wanted) in the next cell; everything else has defaults. To reproduce a
run instead, paste the settings that cell printed in the run's HTML into the cell after it. Everything derived from the
settings is computed after that.

In [ ]:
# parse wrapper-level environment variables: paths, plus the save toggle
SUM_FILE = os.environ.get("SUM_FILE", None)
# SUM_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/2459935/zen.2459935.21341.sum.uvh5'
TOML_FILE = os.environ.get("TOML_FILE", None)
# TOML_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/src/hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml'
SAVE_RESULTS = os.environ.get("SAVE_RESULTS", "TRUE").upper() == "TRUE"
# which [WorkFlow] action is running this notebook
ACTION = os.environ.get("ACTION", "SINGLE_BASELINE_WIENER_INPAINT_NOTEBOOK")

# default names for the files this notebook reads or writes: single-baseline files end in the suffixes, per-night
# products are relative to SUM_FILE's folder ({JD} is the integer JD), and catalogs and sky models are absolute paths
SINGLE_BASELINE_SUFFIX = 'sum.smooth_calibrated.red_avg.uvh5'
INPAINTED_SUFFIX = 'sum.inpainted.uvh5'
WHERE_INPAINTED_SUFFIX = 'sum.where_inpainted.h5'
INPAINT_COVARIANCE_SUFFIX = 'sum.inpaint_covariance.h5'
CORNER_TURN_MAP_FILENAME = 'single_baseline_files/corner_turn_map.yaml'
FLAGS_PI_FRF_FILENAME = 'single_baseline_files/zen.{JD}.flag_waterfall_pI_FRF.h5'  # the night's final flags
SOURCE_CATALOG_FILENAME = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/src/hera_pipelines/pipelines/phase_II/idr1/v1/analysis/source_catalogs/sources_to_filter.yaml'
SOURCE_MODELS_FILENAME = 'single_baseline_files/phased.{JD}.{source}.{model}.uvh5'
SINGLE_BASELINE_SKY_MODEL_FILENAME = '/lustre/aoc/projects/hera/h6c-analysis/abscal_models/h6c_filtered_lst_stack/single_baseline_files/zen.LST.baseline.{ant1}_{ant2}.sum.sky_model.uvh5'  # {ant1}_{ant2} is the redundant group's key; baselines without one get a data-only prior

# shared across the pipeline via [GLOBAL_OPTS]
BAND_SPLIT_FREQ = 100.0  # in MHz; divides the low and high bands (inside the a priori flagged FM gap)
# HERA's correlator design, which sets how its noise departs from the radiometer equation (hera_cal.noise.correlator_noise_covariance)
CORRELATOR = {'n_taps': 4, 'window': 'hamming', 'fwidth': 1.0, 'n_fft': 16384, 'spectra_per_interleave': 2, 'fine_channels_summed': 4}

# default settings, overridden by the TOML's [SINGLE_BASELINE_WIENER_INPAINT_OPTS] section (if given)
AUTO_INPAINT_DELAY = 100.0  # in ns; the smooth 2D model of the autocorrelations that sets the noise weights
NSAMPLES_FILL_DELAY = 100.0  # in ns; the fill of effective nsamples across inpainted gaps (nsamples structure is Tsys-shaped)
EIGENVAL_CUTOFF = 1e-12
CG_TOL = 1e-6  # tolerance of every iterative solve: the 2D DPSS fits (LSQR) and the 2D fill (conjugate gradients)...
CG_ITER_LIM = 3000  # ...and their iteration cap, only a safety net: the 2D fill takes a few hundred iterations (autocorrelations up to ~1700), a cross-correlation's data-only prior up to ~1100
AUTO_FR_SPECTRUM_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/beam_simulation_products/spectra_cache_hera_auto.h5'  # the autocorrelation's fringe-rate spectrum, for the 2D fits' time axis
GAUSS_FIT_BUFFER_CUT = 1e-5
SKY_MODEL_EXTRAP_LIMIT = 0.5  # in units of the model's LST spacing; a data time farther than this from any model LST has no sky model
SKY_MODEL_ADJUST_DELAY = 100.0  # in ns; the sky model is fit to the night by a smooth gain correction out to this delay...
SKY_MODEL_ADJUST_RIDGE = 1.0  # ...and an additive correction out to the horizon, together, with this ridge pulling poorly constrained modes back to no adjustment
DATA_PRIOR_DELAY_BUFFER = 200.0  # in ns; where there is no sky model, the prior is a 2D DPSS fit of the data out to the horizon plus this...
DATA_PRIOR_MIN_DELAY = 500.0  # in ns; ...and at least this far for cross-correlations, to carry the mutual coupling (as the sky model's MIN_FILTER_DELAY)...
DATA_PRIOR_RIDGE = 1e-5  # ...with this light ridge (relative to the mean weight) on a cross-correlation's modes out to the horizon plus the buffer, so a wide gap leaves no mode unconstrained...
DATA_PRIOR_EXTENSION_RIDGE = 0.1  # ...and this one on the modes past the horizon plus the buffer, so they do not pop up in wide gaps
WIENER_TAPER_HOURS = 1.0  # equivalent width of the 2D fill's lag taper in time: its covariance is resolved to about 1 / (this) in fringe rate...
WIENER_TAPER_MHZ = 10.0  # ...and in frequency, i.e. about 1 / (this) in delay
WIENER_THRESHOLD_SIGMA = 3.0  # the covariance keeps modes above this many times its estimate's scatter
SUBTRACT_POLARIZED_SOURCE = True  # subtract per_night_source_filtering's Faraday-rotating (and scintillation) models before inpainting...
SUBTRACT_FG_MODEL = False  # ...and their smooth foreground models too
SUBTRACT_POLARIZED_SOURCE_FROM_SKY_MODEL = False  # also subtract them from the sky model (only right for a sky model made from this night's data)
FR0_FILTER = False  # also write a copy of the inpainted cross-correlations with a fringe-rate-zero notch applied
FR0_HALFWIDTH = 0.01  # in mHz

toml_options = (toml.load(TOML_FILE) if TOML_FILE is not None else {})

# Take the suffixes this notebook reads or writes from [DATA_PRODUCTS], scoped by that
# section's produced_by/consumed_by wiring, and require that wiring to agree exactly with
# the defaults above. That way the arrows drawn on the pipeline flowchart cannot drift from
# what this notebook actually touches: adding an edge there without using the file here (or
# vice versa) fails loudly, in the first cell, rather than silently.
declared_suffixes = {name for name in list(globals()) if name.endswith('_SUFFIX')}
wired_suffixes = set()
for product, spec in toml_options.get('DATA_PRODUCTS', {}).items():
    producers, consumers = (spec.get(key, []) for key in ['produced_by', 'consumed_by'])
    producers = ([producers] if isinstance(producers, str) else producers)
    consumers = ([consumers] if isinstance(consumers, str) else consumers)
    if 'suffix' in spec and (ACTION in producers or ACTION in consumers):
        wired_suffixes.add(f'{product.upper()}_SUFFIX')
        globals()[f'{product.upper()}_SUFFIX'] = spec['suffix']
if toml_options:
    assert wired_suffixes == declared_suffixes, (
        f'[DATA_PRODUCTS] wires {sorted(wired_suffixes)} to {ACTION}, '
        f'but this notebook declares {sorted(declared_suffixes)}.')
# per-night products have no per-file suffix; their [DATA_PRODUCTS] filenames are relative to SUM_FILE's folder
for product in [name[:-len('_FILENAME')] for name in list(globals()) if name.endswith('_FILENAME')]:
    globals()[f'{product}_FILENAME'] = toml_options.get('DATA_PRODUCTS', {}).get(product, {}).get('filename', globals()[f'{product}_FILENAME'])

for toml_section in ['GLOBAL_OPTS', 'SINGLE_BASELINE_WIENER_INPAINT_OPTS']:
    if toml_section in toml_options:
        print(f'Loading overrides from [{toml_section}] in {TOML_FILE}.')
        for key, val in toml_options[toml_section].items():
            globals()[key.upper()] = val

def settings_note():
    '''The settings above as Python assignments. Pasted into a cell right after this one, the printed note reproduces a run
    interactively, since everything derived from the settings is computed after it.'''
    return '\n'.join(f'{setting} = {eval(setting)!r}' for setting in [
        'SUM_FILE', 'TOML_FILE', 'SAVE_RESULTS', 'ACTION', 'SINGLE_BASELINE_SUFFIX', 'INPAINTED_SUFFIX', 'WHERE_INPAINTED_SUFFIX',
        'INPAINT_COVARIANCE_SUFFIX', 'CORNER_TURN_MAP_FILENAME', 'FLAGS_PI_FRF_FILENAME', 'SOURCE_CATALOG_FILENAME', 'SOURCE_MODELS_FILENAME',
        'SINGLE_BASELINE_SKY_MODEL_FILENAME', 'BAND_SPLIT_FREQ', 'CORRELATOR', 'AUTO_INPAINT_DELAY', 'NSAMPLES_FILL_DELAY', 'EIGENVAL_CUTOFF',
        'CG_TOL', 'CG_ITER_LIM', 'AUTO_FR_SPECTRUM_FILE', 'GAUSS_FIT_BUFFER_CUT', 'SKY_MODEL_EXTRAP_LIMIT', 'SKY_MODEL_ADJUST_DELAY',
        'SKY_MODEL_ADJUST_RIDGE', 'DATA_PRIOR_DELAY_BUFFER', 'DATA_PRIOR_MIN_DELAY', 'DATA_PRIOR_RIDGE', 'DATA_PRIOR_EXTENSION_RIDGE',
        'WIENER_TAPER_HOURS', 'WIENER_TAPER_MHZ', 'WIENER_THRESHOLD_SIGMA', 'SUBTRACT_POLARIZED_SOURCE', 'SUBTRACT_FG_MODEL',
        'SUBTRACT_POLARIZED_SOURCE_FROM_SKY_MODEL', 'FR0_FILTER', 'FR0_HALFWIDTH'])

print(settings_note())

In [ ]:
# copy-paste settings note here to interactively override the above defaults or whatever comes from the TOML

In [ ]:
# everything derived from the settings, computed after them so that settings pasted in to reproduce a run carry through:
# the correlator's noise (its variance relative to the radiometer equation, and the correlation between adjacent channels)...
noise_covariance = correlator_noise_covariance('sum', **CORRELATOR)
NOISE_EXCESS, NOISE_CHANNEL_CORRELATION = noise_covariance[0], noise_covariance[1] / noise_covariance[0]
# ...the polarized-source model components to subtract, and the night's inputs
MODEL_COMPONENTS = (["fg_model"] if SUBTRACT_FG_MODEL else []) + ["rm_model", "scint_model"]
jdstr = re.search(r'zen\.(\d+)\.', os.path.basename(SUM_FILE)).group(1)
CORNER_TURN_MAP_YAML = os.path.join(os.path.dirname(SUM_FILE), CORNER_TURN_MAP_FILENAME)
FINAL_FLAG_FILE = os.path.join(os.path.dirname(SUM_FILE), FLAGS_PI_FRF_FILENAME.format(JD=jdstr))
for name in ['NOISE_EXCESS', 'NOISE_CHANNEL_CORRELATION', 'MODEL_COMPONENTS', 'CORNER_TURN_MAP_YAML', 'FINAL_FLAG_FILE']:
    print(f'{name} = {eval(name)}')

add_to_history = ('Produced by the single_baseline_wiener_inpaint notebook with the following settings:\n' + settings_note() + '\n'
                  + 'and the following environment:\n' + '=' * 65 + '\n' + os.popen('conda env export').read() + '=' * 65)

## Preliminaries

In [ ]:
with open(CORNER_TURN_MAP_YAML, 'r') as file:
    corner_turn_map = yaml.unsafe_load(file)

# the map is keyed by the night's redundantly averaged files; this job's share is under the one with SUM_FILE's JD
jd_str = re.search(r'zen\.(\d+\.\d+)\.', os.path.basename(SUM_FILE)).group(1)
single_bl_files = next((outfiles for red_avg_file, outfiles in corner_turn_map['files_to_outfiles_map'].items()
                        if f'zen.{jd_str}.' in os.path.basename(red_avg_file)), [])
print(f'The corner-turn map assigns {len(single_bl_files)} single-baseline files to zen.{jd_str}.')

# the night's final flags: the single-baseline files carry only the flags the calibration left
prior_flags = np.all(UVFlag(FINAL_FLAG_FILE).flag_array, axis=-1)
print(f'Final flags from {FINAL_FLAG_FILE}: {np.mean(prior_flags):.3%} flagged.')

In [ ]:
if SUBTRACT_POLARIZED_SOURCE:
    with open(SOURCE_CATALOG_FILENAME) as f:
        source_config = yaml.safe_load(f)

    model_specs = []
    for source in source_config["sources"]:
        fsname = source["name"].replace(" ", "_").replace("-", "_")
        for model_name in MODEL_COMPONENTS:
            model_file = os.path.join(os.path.dirname(SUM_FILE), SOURCE_MODELS_FILENAME.format(JD=jdstr, source=fsname, model=model_name))
            if os.path.isfile(model_file):
                model_specs.append((model_file, model_name, source))

    print(f"Using model components: {MODEL_COMPONENTS}")
    print(f"Found {len(model_specs)} model files" + (" (no polarized source was modeled on this night)." if len(model_specs) == 0 else ""))
    for model_file, model_name, source in model_specs:
        bands = {(False, False): "nowhere", (True, False): "below FM only", (False, True): "above FM only", (True, True): "in both bands"}
        print(f"  {os.path.basename(model_file)}: subtracted {bands[(source['subtract_below_fm'], source['subtract_above_fm'])]}")

## Functions for main loop

In [ ]:
FR_CENTER_AND_HW_CACHE = {}

def cache_fr_center_and_hw(hd, antpair, tslice, band):
    '''Figure out the range of FRs in Hz spanned for a given band and tslice, buffered by the size of the autocorrelation FR kernel,
    and stores the value in FR_CENTER_AND_HW_CACHE (if it hasn't already been computed.'''
    if (tslice is not None) and (band is not None) and ((antpair, tslice, band) not in FR_CENTER_AND_HW_CACHE):
        # calculate fringe rate center and half-width and then update cache
        fr_buffer = get_FR_buffer_from_spectra(AUTO_FR_SPECTRUM_FILE, hd.times[tslice], hd.freqs[band], 
                                               gauss_fit_buffer_cut=GAUSS_FIT_BUFFER_CUT)
        hd_here = hd.select(inplace=False, frequencies=hd.freqs[band])
        fr_center = list(sky_frates(hd_here)[0].values())[0] / 1e3  # converts to Hz
        fr_hw = (list(sky_frates(hd_here)[1].values())[0] + fr_buffer) / 1e3    
        FR_CENTER_AND_HW_CACHE[(antpair, tslice, band)] = fr_center, fr_hw

In [ ]:
def get_ip_nsamples(nsamples, flags, tslices, bands):
    '''Effective nsamples at the flagged cells of every integration with some unflagged data in the band, for the inpainting
    weights and, at inpainted cells, for the output. Phase II's effective nsamples vary with frequency (they carry the
    autocorrelations' spectra) and are 0 wherever every member of the redundant group was flagged, so unlike H6C's per-band
    counts they must be filled in: by a DPSS fit (NSAMPLES_FILL_DELAY) along frequency of the integration's unflagged
    nsamples, clipped to their range. So LST stacking weights an inpainted model by the depth the night would have had
    there. Unflagged cells, and integrations fully flagged in the band (never inpainted, zero weight), are unchanged.'''
    ip_nsamples = copy.deepcopy(nsamples)
    for bl in nsamples:
        for tslice, band in zip(tslices[bl], bands[bl]):
            if (tslice is None) or (band is None):
                continue
            rows = np.where(~np.all(flags[bl][tslice, band], axis=1))[0] + tslice.start
            if len(rows) == 0:
                continue
            wgts = np.where(flags[bl][rows][:, band], 0, 1).astype(float)
            fit, _, _ = fourier_filter(nsamples.freqs[band], nsamples[bl][rows][:, band].astype(complex), wgts=wgts,   # the fit is complex
                                       filter_centers=[0], filter_half_widths=[NSAMPLES_FILL_DELAY * 1e-9], mode='dpss_solve',
                                       eigenval_cutoff=[EIGENVAL_CUTOFF], suppression_factors=[EIGENVAL_CUTOFF],
                                       max_contiguous_edge_flags=len(nsamples.freqs), filter_dims=1)
            for k, tind in enumerate(rows):
                unflagged = nsamples[bl][tind, band][wgts[k] > 0]
                fill = np.clip(fit[k].real, np.min(unflagged), np.max(unflagged))
                ip_nsamples[bl][tind, band] = np.where(flags[bl][tind, band], fill, nsamples[bl][tind, band])
    return ip_nsamples

In [ ]:
def get_weights_for_inpainting(data, flags, tslices, bands, ip_nsamples, ip_autos, auto_flags):
    '''Get inverse noise variance weights for inpainting. These come in two flavors:
        * weights_before_ip: has 0s wherever the data or autos are flagged
        * weights_after_ip: uses inpainted autos for "noise," so only has 0s wherever the
            autos weren't inpainted (in practice, nowhere in the bands/tslice of interest) and in
            integrations fully flagged in the band, which are never inpainted and whose (possibly
            zero) nsamples are never filled: nothing anchors the 2D fits there.
    '''
    weights_before_ip = {}
    weights_after_ip = {}
    for bl in data:
        ant1, ant2 = utils.split_bl(bl)
        
        auto_bl_1 = [k for k in ip_autos if k[2] == utils.join_pol(ant1[1], ant1[1])][0]
        auto_bl_2 = [k for k in ip_autos if k[2] == utils.join_pol(ant2[1], ant2[1])][0]
        with np.errstate(divide='ignore', invalid='ignore'):  # nsamples can be 0 in fully flagged integrations
            noise = (np.abs(ip_autos[auto_bl_1] * ip_autos[auto_bl_2]) / (ip_nsamples[bl] * dt * df))**.5

        # assign non-zero weights only in the bands/tslices of interest
        weights_before_ip[bl] = np.zeros_like(data[bl], dtype=float)
        weights_after_ip[bl] = np.zeros_like(data[bl], dtype=float)
        for tslice, band in zip(tslices[bl], bands[bl]):
            if (tslice is None) or (band is None):
                continue
            
            non_finite_ip_auto = (~np.isfinite(ip_autos[auto_bl_1][tslice, band])) 
            non_finite_ip_auto |= (~np.isfinite(ip_autos[auto_bl_2][tslice, band]))
            fully_flagged = np.all(flags[bl][tslice, band], axis=1, keepdims=True)
            weights_after_ip[bl][tslice, band] = np.where(non_finite_ip_auto | fully_flagged, 0, noise[tslice, band]**-2)

            flags_here = auto_flags[auto_bl_1][tslice, band] | auto_flags[auto_bl_2][tslice, band] 
            flags_here |= flags[bl][tslice, band] | non_finite_ip_auto
            weights_before_ip[bl][tslice, band] = np.where(flags_here, 0, noise[tslice, band]**-2)

        # renormalize
        for wgts in [weights_after_ip[bl], weights_before_ip[bl]]:
            if np.any(wgts > 0):
                wgts /= np.mean(wgts[wgts > 0])
    return weights_before_ip, weights_after_ip

In [ ]:
def fit_2D_DPSS(data, weights, filter_delay, tslices, bands, **kwargs):
    '''Fit a 2D DPSS model to all the baselines in data. The time-dimension is based on sky FRs
    and the FR spectrum of the autos. fr_centers and fr_hws are drawn from FR_CENTER_AND_HW_CACHE.
    Used for the smooth autocorrelation model and for an autocorrelation's data-only prior.
    
    Arguments:
        data: datacontainer mapping baselines to complex visibility waterfalls
        weights: datacontainer mapping baselines to real weight waterfalls. 
        filter_delay: maximum delay in ns for the 2D filter
        tslices: dictionary mapping bl to time slices corresponding to low and high bands
        bands: dictionary mapping bl to low band and high band frequency slices
        kwargs: kwargs to pass into sparse_linear_fit_2D()
    
    Returns:
        dpss_fit: datacontainer mapping baselines to 2D DPSS models
    '''
    dpss_fit = copy.deepcopy(data)
    for bl in data.keys():
        # set to all nans by default
        dpss_fit[bl] *= np.nan

        for tslice, band in zip(tslices[bl], bands[bl]):
            if (tslice is None) or (band is None) or np.all(weights[bl][tslice, band] == 0):
                continue

            # perform 2D DPSS filter    
            fr_center, fr_hw = FR_CENTER_AND_HW_CACHE[(bl[0:2], tslice, band)]
            time_filters, _ = dpss_operator((data.times[tslice] - data.times[tslice][0]) * 3600 * 24, 
                                            [fr_center], [fr_hw], eigenval_cutoff=[EIGENVAL_CUTOFF])
            freq_filters, _ = dpss_operator(data.freqs[band], [0.0], [filter_delay / 1e9], eigenval_cutoff=[EIGENVAL_CUTOFF])
            
            fit, meta = sparse_linear_fit_2D(
                data=data[bl][tslice, band],
                weights=weights[bl][tslice, band],
                axis_1_basis=time_filters,
                axis_2_basis=freq_filters,
                precondition_solver=True,
                iter_lim=CG_ITER_LIM,
                **kwargs,
            )
            if meta.get('istop') == 7:   # LSQR's stop code 7: it reached iter_lim without meeting atol/btol
                print(f'\tWARNING: the 2D DPSS fit of {bl} at {data.freqs[band][0] / 1e6:.0f}-{data.freqs[band][-1] / 1e6:.0f} MHz stopped '
                      f'at the iteration cap ({CG_ITER_LIM}) without converging.')
            dpss_fit[bl][tslice, band] = time_filters.dot(fit).dot(freq_filters.T)
            
    return dpss_fit

In [ ]:
def four_pol_inpainting_figure(ip_data, flags, ip_flags, close=False):
    '''Plots all phase and amplitude waterfalls before and after inpainting for all 4 polarizations in the data.'''
    fig, axes = plt.subplots(4, 4, figsize=(16, 30), sharex=True, sharey=True, dpi=200, gridspec_kw={'wspace': 0.02, 'hspace': 0.01})
    
    vmin = np.nanmin([np.where(~flags[bl] & (np.abs(ip_data[bl]) > 0), np.abs(ip_data[bl]), np.nan) for bl in ip_data])
    vmax = np.nanmax([np.where(~flags[bl] & np.isfinite(ip_data[bl]), np.abs(ip_data[bl]), np.nan) for bl in ip_data])
    lst_grid = ip_data.lsts * 12 / np.pi
    lst_grid[lst_grid > lst_grid[-1]] -= 24
    frac_jds = ip_data.times - int(ip_data.times[0])
    extent = [ip_data.freqs[0] / 1e6, ip_data.freqs[-1] / 1e6, frac_jds[-1], frac_jds[0]]
    
    for row, bl in zip(axes, data):
    
        row[0].imshow(np.where(flags[bl], np.nan, np.angle(ip_data[bl])), aspect='auto', interpolation='none', cmap='twilight', extent=extent)
        row[1].imshow(np.where(ip_flags[bl], np.nan, np.angle(ip_data[bl])), aspect='auto', interpolation='none', cmap='twilight', extent=extent)
        row[2].imshow(np.where(flags[bl], np.nan, np.abs(ip_data[bl])), aspect='auto', interpolation='none', norm=matplotlib.colors.LogNorm(vmin=vmin, vmax=vmax), extent=extent)
        im = row[3].imshow(np.where(ip_flags[bl], np.nan, np.abs(ip_data[bl])), aspect='auto', interpolation='none', norm=matplotlib.colors.LogNorm(vmin=vmin, vmax=vmax), extent=extent)
        
        row[0].set_ylabel(f'JD - {int(ip_data.times[0])}')
        for ax in row:
            ax.tick_params(axis='x', direction='in')
    
        row[0].text(0.02, 0.99, bl, transform=row[0].transAxes, ha='left', va='top', fontsize=12, color='white',
                bbox=dict(facecolor='black', alpha=0.5, pad=2))

    # Add LST right axis on rightmost column of last row
    ax2 = axes[-1][-1].twinx()
    ax2.set_ylim(lst_grid[-1], lst_grid[0])
    mod24 = lambda x, _: f"{x % 24:.1f}"
    ax2.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(mod24))
    ax2.set_ylabel('LST (hours)')
    
    for ax in axes[-1]:
        ax.set_xlabel('Frequency (MHz)')
    
    plt.tight_layout()   
    plt.colorbar(im, ax=axes[0], location='top', label='|V| (Jy)', aspect=50)
    
    if close:
        plt.close(fig)
    return fig

In [ ]:
def regenerate_P(cov):
    """The thresholded power spectrum P(fringe rate, delay) the 2D fill used, on its zero-padded (2 nt, 2 nf) grid, rebuilt from what
    the covariance file stores (see write_inpaint_covariance): the tapered lag function's non-negative time lags, completed by
    Hermitian symmetry, Fourier transformed, less the noise floor, and thresholded."""
    nt, nf, Kt, Kf = cov['nt'], cov['nf'], cov['Kt'], cov['Kf']
    lags_f = np.concatenate([np.arange(0, Kf + 1), np.arange(2 * nf - Kf, 2 * nf)])
    c = np.zeros((2 * nt, 2 * nf), dtype=complex)
    c[np.arange(Kt + 1)[:, None], lags_f] = cov['half_lag_function']
    c[-np.arange(1, Kt + 1)[:, None] % (2 * nt), -lags_f % (2 * nf)] = np.conj(cov['half_lag_function'][1:])   # c(-t, -f) = c(t, f)*
    P_raw = np.real(sfft.fft2(c, workers=FFT_WORKERS))
    return np.where(P_raw - cov['noise_floor'] > WIENER_THRESHOLD_SIGMA * cov['sigma_P'], P_raw - cov['noise_floor'], 0)


def covariance_figure(covariances, antpair, antpos, close=False):
    """Figure 2: the ee and nn covariances the 2D fill learned for this baseline (one per band), rebuilt from what the covariance file
    stores. Top: the correlation of the covariance it used, C, against time and frequency lag, out to the lag tapers' zeros. Middle
    (low band) and bottom (high band): its Fourier transform (P here, C-tilde in the markdown) summed over delay against fringe rate,
    and over fringe rate against delay, out to the Nyquist rates. Each column is a Fourier pair."""
    fig, axes = plt.subplots(3, 2, figsize=(16, 13), dpi=150, constrained_layout=True)
    band_names = sorted({key.split('/')[1] for key in covariances}, key=lambda name: float(name.split('-')[0]))
    sky_windows = {}   # per row: the sky's fringe rates (with the autocorrelation's buffer), the same for every polarization
    for key in sorted(covariances, key=lambda key: (key.split('/')[0], float(key.split('/')[1].split('-')[0]))):
        pol, band_name = key.split('/')
        if pol not in ['ee', 'nn']:
            continue
        cov = covariances[key]
        P = regenerate_P(cov)
        low = float(band_name.split('-')[0]) < BAND_SPLIT_FREQ
        color = {'ee': 'C0', 'nn': 'C1'}[pol]
        if not np.any(P > 0):
            axes[0, 1].plot([], [], color=color, ls='-' if low else '--', label=f'{pol}, {band_name}: no modes above the threshold')
            continue
        # the covariance's cuts through zero lag are the 1D inverse transforms of P's sums, each up to the same constant as at zero lag
        P_fr, P_tau = P.sum(axis=1), P.sum(axis=0)
        C_t, C_f = sfft.ifft(P_fr)[:cov['Kt'] + 1], sfft.ifft(P_tau)[:cov['Kf'] + 1]
        axes[0, 0].plot(np.arange(cov['Kt'] + 1) * dt / 60, np.abs(C_t) / C_t[0].real, color=color, ls='-' if low else '--', lw=1.5)
        axes[0, 1].plot(np.arange(cov['Kf'] + 1) * df / 1e6, np.abs(C_f) / C_f[0].real, color=color, ls='-' if low else '--', lw=1.5,
                        label=f'{pol}, {band_name}: $C(0, 0)$ = {np.mean(P) / cov["mean_noise"]:.2g}× the noise variance')
        # P's sums as dots: the modes below the threshold are zero, which a log axis leaves out
        row = 1 if low else 2
        axes[row, 0].semilogy(np.fft.fftshift(np.fft.fftfreq(2 * cov['nt'], dt)) * 1e3, np.fft.fftshift(P_fr) / np.max(P_fr), '.',
                              color=color, ms=2, label=pol)
        axes[row, 1].semilogy(np.fft.fftshift(np.fft.fftfreq(2 * cov['nf'], df)) * 1e9, np.fft.fftshift(P_tau) / np.max(P_tau), '.',
                              color=color, ms=2, label=pol)
        window = FR_CENTER_AND_HW_CACHE.get((antpair, slice(*cov['tslice']), slice(*cov['band'])))
        if window is not None:
            sky_windows[row] = window
    horizon_ns = np.linalg.norm(antpos[antpair[1]] - antpos[antpair[0]]) / constants.c * 1e9
    axes[0, 0].set(xlabel=r'time lag $\Delta t$ (minutes)', ylabel=r'$|C(\Delta t, 0)|\ /\ C(0, 0)$', ylim=(0, 1.05))
    axes[0, 1].set(xlabel=r'frequency lag $\Delta\nu$ (MHz)', ylabel=r'$|C(0, \Delta\nu)|\ /\ C(0, 0)$', ylim=(0, 1.05))
    axes[0, 1].legend(fontsize=9)
    for row, band_name in zip([1, 2], band_names if len(band_names) == 2 else [None, None]):
        if row in sky_windows:
            for sign in [-1, 1]:
                axes[row, 0].axvline((sky_windows[row][0] + sign * sky_windows[row][1]) * 1e3, color='0.4', lw=0.8,
                                     label=('sky fringe rates' if sign == 1 else None))
        if horizon_ns > 0:
            for sign in [-1, 1]:
                axes[row, 1].axvline(sign * horizon_ns, color='0.4', lw=1, ls=':', label=('horizon delay' if sign == 1 else None))
        axes[row, 0].set(xlabel=r'fringe rate $f$ (mHz)', ylabel=r'$\sum_\tau \tilde{C}(f, \tau)$, normalized to its peak', ylim=(1e-4, 2),
                         xlim=(-0.5e3 / dt, 0.5e3 / dt))   # out to the Nyquist fringe rate
        axes[row, 1].set(xlabel=r'delay $\tau$ (ns)', ylabel=r'$\sum_f \tilde{C}(f, \tau)$, normalized to its peak', ylim=(1e-4, 2),
                         xlim=(-0.5e9 / df, 0.5e9 / df))   # out to the Nyquist delay
        for ax in axes[row]:
            ax.set_title(band_name if band_name is not None else ('low band' if row == 1 else 'high band'))
            if ax.get_legend_handles_labels()[0]:
                ax.legend(fontsize=9, markerscale=4)
    fig.suptitle(f'Covariance learned by the 2D Wiener fill for baseline {tuple(int(a) for a in antpair)}')
    if close:
        plt.close(fig)
    return fig

In [ ]:
def sky_model_adjustment_figure(data, G, R, sky_model_flags, ip_nsamples, close=False):
    """Figure 1: how adjust_sky_model_to_data adjusted the sky model to the night, as sky_model * (1 + G) + R, one row per polarization. The
    columns are the gain applied to the sky model, |1 + G| and arg(1 + G), and the additive correction, |R| in units of the noise each
    cell has (or would have had, where flagged) and arg(R). Blank where the sky model is flagged or nothing was fit."""
    fig, axes = plt.subplots(len(data), 4, figsize=(16, 3 * len(data)), sharex=True, sharey=True, dpi=100, squeeze=False,
                             constrained_layout=True)
    # every 4th integration and every 2nd channel: each panel is a few hundred pixels across and G and R are smooth by construction,
    # so they look the same, and the figure does not hold full-size copies of every panel until it is displayed
    cells = (slice(None, None, 4), slice(None, None, 2))
    frac_jds = data.times[cells[0]] - int(data.times[0])
    lst_grid = data.lsts[cells[0]] * 12 / np.pi
    lst_grid[lst_grid > lst_grid[-1]] -= 24
    extent = [data.freqs[cells[1]][0] / 1e6, data.freqs[cells[1]][-1] / 1e6, frac_jds[-1], frac_jds[0]]
    gain, snr, R_phase = {}, {}, {}
    for bl in data:
        shown = np.isfinite(G[bl][cells]) & ~sky_model_flags[bl][cells]
        # the noise the 2D fill assumes (see wiener_inpaint): the radiometer equation with the smooth autos and the filled effective
        # nsamples, times the correlator's excess
        auto_1, auto_2 = [auto_fit[next(k for k in auto_fit if k[2] == utils.join_pol(ant[1], ant[1]))][cells] for ant in utils.split_bl(bl)]
        with np.errstate(divide='ignore', invalid='ignore'):   # nsamples can be 0 in fully flagged integrations
            noise = np.sqrt(NOISE_EXCESS * np.abs(auto_1 * auto_2) / (ip_nsamples[bl][cells] * dt * df))
        gain[bl] = np.where(shown, 1 + G[bl][cells], np.nan)
        snr[bl] = np.where(shown & np.isfinite(noise) & (noise > 0), np.abs(R[bl][cells]) / noise, np.nan)
        R_phase[bl] = np.where(shown, np.angle(R[bl][cells]), np.nan)

    # color scales shared by every polarization: |1 + G| symmetric about 1, arg(1 + G) about 0, and |R| / sigma_N over three decades
    gains = np.concatenate([g[np.isfinite(g)] for g in gain.values()])
    snrs = np.concatenate([s[np.isfinite(s)] for s in snr.values()])
    amp_range = np.exp(max(np.percentile(np.abs(np.log(np.abs(gains))), 99.5), 0.01))
    phase_range = min(max(np.percentile(np.abs(np.angle(gains)), 99.5), 0.01), np.pi)
    snr_max = (max(np.percentile(snrs, 99.5), 1) if snrs.size else 1)
    for row, bl in zip(axes, data):
        ims = [row[0].imshow(np.abs(gain[bl]), cmap='RdBu_r', norm=matplotlib.colors.TwoSlopeNorm(1, 1 / amp_range, amp_range),
                             aspect='auto', interpolation='none', extent=extent),
               row[1].imshow(np.angle(gain[bl]), cmap='RdBu_r', vmin=-phase_range, vmax=phase_range, aspect='auto', interpolation='none',
                             extent=extent),
               row[2].imshow(snr[bl], cmap='inferno', norm=matplotlib.colors.LogNorm(snr_max / 1e3, snr_max), aspect='auto',
                             interpolation='none', extent=extent),
               row[3].imshow(R_phase[bl], cmap='twilight', vmin=-np.pi, vmax=np.pi,
                             aspect='auto', interpolation='none', extent=extent)]
        row[0].set_ylabel(f'JD - {int(data.times[0])}')
        lst_axis = row[-1].twinx()
        lst_axis.set_ylim(lst_grid[-1], lst_grid[0])
        lst_axis.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'{x % 24:.1f}'))
        lst_axis.set_ylabel('LST (hours)')
        row[0].text(0.02, 0.97, bl[2], transform=row[0].transAxes, ha='left', va='top', color='white',
                    bbox=dict(facecolor='black', alpha=0.5, pad=2))
    for ax in axes[-1]:
        ax.set_xlabel('Frequency (MHz)')
    for col, (im, label) in enumerate(zip(ims, [r'$|1 + G|$', r'$\arg(1 + G)$ (radians)', r'$|R|\ /\ \sigma_N$', r'$\arg(R)$ (radians)'])):
        fig.colorbar(im, ax=axes[:, col], location='top', label=label, aspect=30)
    fig.suptitle(f'Adjustment of the sky model to the night for baseline {tuple(int(a) for a in data.antpairs().pop())}: '
                 'sky model × (1 + G) + R')
    if close:
        plt.close(fig)
    return fig

In [ ]:
def subtract_polarized_models(data, flags, model_specs, extra_flags=None, baseline=None):
    '''Subtract each polarized source model (file, component, catalog entry) from the data in place, in the bands the
    catalog entry's subtract_below_fm / subtract_above_fm flags select, zeroing the model wherever the data or the
    model are flagged.'''
    for model_file, model_name, source in model_specs:
        if not (source['subtract_below_fm'] or source['subtract_above_fm']):
            continue
        freq_range = (0 if source['subtract_below_fm'] else BAND_SPLIT_FREQ * 1e6, np.inf if source['subtract_above_fm'] else BAND_SPLIT_FREQ * 1e6)
        polfilt.subtract_polarized_source_model(data, flags, model_file, baseline=baseline, extra_flags=extra_flags, freq_range=freq_range)

In [ ]:
sky_model_group_keys = red_groups.RedundantGroups.from_antpos(io.HERAData(single_bl_files[0]).antpos) if len(single_bl_files) > 0 else None
sky_model_dir, sky_model_pattern = os.path.split(SINGLE_BASELINE_SKY_MODEL_FILENAME)
sky_model_files = {}
for sky_model_file in glob.glob(os.path.join(sky_model_dir, sky_model_pattern.format(ant1='*', ant2='*'))):
    match = re.search(sky_model_pattern.replace('.', r'\.').format(ant1=r'(\d+)', ant2=r'(\d+)'), os.path.basename(sky_model_file))
    if match:
        sky_model_files[(int(match.group(1)), int(match.group(2)))] = sky_model_file
print(f'Found {len(sky_model_files)} single-baseline sky models in {sky_model_dir}.')


def find_sky_model_file(antpair):
    '''The sky model of the redundant group antpair belongs to, in either orientation, or None.'''
    for ap in sky_model_group_keys.get_red(antpair):
        for candidate in [tuple(ap), tuple(ap)[::-1]]:
            if candidate in sky_model_files:
                return sky_model_files[candidate]
    return None


def load_sky_model(hd, data, sky_model_file):
    '''The sky model matched to hd's times (each to the model integration nearest in LST, within SKY_MODEL_EXTRAP_LIMIT
    model integrations) and rephased to its exact LSTs, keyed like data, with the model's flags. Integrations the model
    does not cover come back fully flagged, so that they count as uncovered and get the data-only prior. Returns
    (None, None) if the model lacks a polarization or covers none of the night's integrations.'''
    hdm = io.HERAData(sky_model_file)
    if any(pol not in hdm.pols for pol in data.pols()):
        print(f'\tThe sky model lacks the {[pol for pol in sorted(data.pols()) if pol not in hdm.pols]} polarization(s).')
        return None, None
    all_model_times, all_model_lsts = abscal.get_all_times_and_lsts(hdm, unwrap=True)
    d2m_time_map = abscal.get_d2m_time_map(data.times, np.unwrap(data.lsts), all_model_times, all_model_lsts,
                                           extrap_limit=SKY_MODEL_EXTRAP_LIMIT)
    matched = np.array([d2m_time_map[time] is not None for time in data.times])
    if not np.any(matched):
        print(f'\tThe sky model has no integration within {SKY_MODEL_EXTRAP_LIMIT} of its LST spacing of any of the night\'s.')
        return None, None
    if not np.all(matched):
        print(f'\tThe sky model has no integration within {SKY_MODEL_EXTRAP_LIMIT} of its LST spacing of {np.sum(~matched)} of '
              f'the night\'s {len(matched)} integrations.')
    model_times = np.array([d2m_time_map[time] for time in data.times[matched]])
    model, model_flags, _ = io.partial_time_io(hdm, np.unique(model_times), polarizations=sorted(data.pols()))
    row = {time: i for i, time in enumerate(model.times)}
    rows = np.array([row[time] for time in model_times])
    sky_model, sky_model_flags = {}, {}
    for bl in data:
        sky_model[bl] = np.zeros_like(data[bl])
        sky_model_flags[bl] = np.ones_like(data[bl], dtype=bool)
        sky_model[bl][matched], sky_model_flags[bl][matched] = model[bl][rows], model_flags[bl][rows]
    sky_model = datacontainer.DataContainer(sky_model)
    sky_model.freqs, sky_model.times, sky_model.lsts, sky_model.antpos = data.freqs, data.times, data.lsts, data.antpos
    blvecs = {bl: data.antpos[bl[0]] - data.antpos[bl[1]] for bl in data}
    dlst = np.zeros(len(data.times))
    dlst[matched] = data.lsts[matched] - model.lsts[rows]
    utils.lst_rephase(sky_model, blvecs, data.freqs, dlst, lat=hd.telescope.location.lat.deg, inplace=True)
    return sky_model, datacontainer.DataContainer(sky_model_flags)

### Fitting the sky model to the night

The sky model $m(t, \nu)$ of the baseline's redundant group (an LST-stacked, 2D-filtered average of many nights) is adjusted to this
night as
$$\hat m(t, \nu) = m(t, \nu)\,\big[1 + G(t, \nu)\big] + R(t, \nu),$$
with a smooth complex gain correction $G$ and an additive correction $R$, each a 2D DPSS model,
$$G = \sum_{k,l} a_{kl}\, T_k(t)\, F^g_l(\nu), \qquad R = \sum_{k,l} b_{kl}\, T_k(t)\, F^r_l(\nu).$$
The time modes $T_k$ span the sky's fringe rates (with a buffer from the autocorrelation's fringe-rate spectrum); the frequency modes
$F^g_l$ span delays out to `SKY_MODEL_ADJUST_DELAY` and $F^r_l$ out to the horizon delay $\tau_h = |\vec b|/c$. Both corrections are fit
at once, by LSQR, over the cells unflagged in both the data and the model:
$$\min_{a,\,b}\; \sum_{t,\nu} w\,\big|V - m - m\,G - R\big|^2 + \lambda_g \lVert a \rVert^2 + \lambda_r \lVert b \rVert^2,
\qquad \lambda_g = \rho \,\langle w |m|^2 \rangle, \quad \lambda_r = \rho\, \langle w \rangle,$$
where $w$ are the noise weights and $\rho$ = `SKY_MODEL_ADJUST_RIDGE`.
* **The ridges:** each is relative to the weight its correction sees, so a mode with a fraction $e$ of its weighted energy on observed
  cells is shrunk by about $e / (e + \rho)$, and modes the data constrain poorly relax to no adjustment ($G = 0$, $R = 0$).
* **One damping in LSQR:** LSQR applies one damping to every coefficient, so $G$'s coefficients are solved as $a = s\,x_g$ with
  $s = \sqrt{\lambda_r / \lambda_g}$.

In [ ]:
def adjust_sky_model_to_data(data, flags, sky_model, sky_model_flags, weights, tslices, bands):
    '''Fit the sky model to this night's unflagged data as sky_model * (1 + G) + R, where G is a smooth complex gain
    correction (SKY_MODEL_ADJUST_DELAY in delay) and R an additive correction (out to the baseline's horizon delay), both 2D DPSS
    models spanning the sky fringe rates in time. G and R are fit together by LSQR over the cells unflagged in both data and
    sky model, with a ridge (SKY_MODEL_ADJUST_RIDGE, relative to the data's weight on each mode) that pulls the modes the data
    constrain poorly back toward no adjustment. Returns the adjusted sky model (the sky model itself outside the fits), and G and R
    (np.nan outside the fits).'''
    adjusted = copy.deepcopy(sky_model)
    G = {bl: np.full(data[bl].shape, np.nan, dtype=complex) for bl in data}
    R = {bl: np.full(data[bl].shape, np.nan, dtype=complex) for bl in data}
    antpair = data.antpairs().pop()
    horizon_ns = np.linalg.norm(data.antpos[antpair[1]] - data.antpos[antpair[0]]) / constants.c * 1e9
    for bl in data:
        for tslice, band in zip(tslices[bl], bands[bl]):
            if (tslice is None) or (band is None):
                continue
            wgts = np.where(~flags[bl][tslice, band] & ~sky_model_flags[bl][tslice, band] & (sky_model[bl][tslice, band] != 0),
                            weights[bl][tslice, band], 0)
            if not np.any(wgts > 0):
                continue
            # G = T a F_g^T and R = T b F_r^T, with DPSS bases in time (T, spanning the sky fringe rates) and in frequency (F_g out to
            # SKY_MODEL_ADJUST_DELAY, F_r out to the horizon), so sky_model * (1 + G) + R is linear in the coefficients a and b
            fr_center, fr_hw = FR_CENTER_AND_HW_CACHE[(bl[0:2], tslice, band)]
            time_filters, _ = dpss_operator((data.times[tslice] - data.times[tslice][0]) * 3600 * 24, [fr_center], [fr_hw], eigenval_cutoff=[EIGENVAL_CUTOFF])
            g_filters, _ = dpss_operator(data.freqs[band], [0.0], [SKY_MODEL_ADJUST_DELAY / 1e9], eigenval_cutoff=[EIGENVAL_CUTOFF])
            r_filters, _ = dpss_operator(data.freqs[band], [0.0], [horizon_ns / 1e9], eigenval_cutoff=[EIGENVAL_CUTOFF])
            mdl = np.where(wgts > 0, sky_model[bl][tslice, band], 0)
            sqrt_w = np.sqrt(wgts)
            # each correction gets the ridge its own sequential fit would have, relative to the weight on its basis: R's columns
            # carry w, G's carry w |sky_model|^2 (G multiplies the sky model). LSQR minimizes |A x - y|^2 + damp^2 |x|^2 with one damp
            # for every coefficient, so G's are fit as a = scale * x_g, which turns damp_r2 |x_g|^2 into damp_g2 |a|^2
            damp_g2, damp_r2 = SKY_MODEL_ADJUST_RIDGE * np.mean((wgts * np.abs(mdl)**2)[wgts > 0]), SKY_MODEL_ADJUST_RIDGE * np.mean(wgts[wgts > 0])
            scale = np.sqrt(damp_r2 / damp_g2)
            nt, ng = time_filters.shape[1], time_filters.shape[1] * g_filters.shape[1]

            # x stacks G's and R's coefficients, each a (time mode x frequency mode) matrix, flattened; A x is the weighted model,
            # sqrt(w) * (scale * sky_model * T x_g F_g^T + T x_r F_r^T), on every cell of the block
            def matvec(x):
                return (sqrt_w * (scale * mdl * (time_filters @ x[:ng].reshape(nt, -1) @ g_filters.T)
                                  + time_filters @ x[ng:].reshape(nt, -1) @ r_filters.T)).ravel()

            # LSQR also needs the adjoint, A^H y: the same products, transposed and conjugated
            def rmatvec(y):
                y = sqrt_w * y.reshape(wgts.shape)
                return np.concatenate([(scale * time_filters.T.conj() @ (np.conj(mdl) * y) @ g_filters.conj()).ravel(),
                                       (time_filters.T.conj() @ y @ r_filters.conj()).ravel()])

            A = spla.LinearOperator((wgts.size, ng + nt * r_filters.shape[1]), matvec=matvec, rmatvec=rmatvec, dtype=complex)
            # the target is the weighted difference sqrt(w) * (data - sky_model), which G and R together must explain
            x = spla.lsqr(A, (sqrt_w * np.where(wgts > 0, data[bl][tslice, band] - mdl, 0)).ravel(), damp=np.sqrt(damp_r2),
                          atol=CG_TOL, btol=CG_TOL, iter_lim=CG_ITER_LIM)[0]
            G[bl][tslice, band] = scale * time_filters @ x[:ng].reshape(nt, -1) @ g_filters.T
            R[bl][tslice, band] = time_filters @ x[ng:].reshape(nt, -1) @ r_filters.T
            adjusted[bl][tslice, band] = sky_model[bl][tslice, band] * (1 + G[bl][tslice, band]) + R[bl][tslice, band]
    return adjusted, G, R

### The data-only prior

Where there is no sky model, where it is flagged, and for every autocorrelation, the prior is a 2D DPSS fit of the night's own data,
$$f(t, \nu) = \sum_{k,l} x_{kl}\, T_k(t)\, F_l(\nu),$$
with the same time modes and frequency modes out to $\tau_h$ + `DATA_PRIOR_DELAY_BUFFER` ($\tau_h = 0$ for an autocorrelation) and, for a
cross-correlation, at least `DATA_PRIOR_MIN_DELAY`, so that the prior carries the mutual coupling. A cross-correlation's fit is
$$\min_x\; \sum_{t,\nu} w\,|V - f|^2 + \sum_{k,l} \lambda_l\, |x_{kl}|^2, \qquad
\lambda_l = \langle w \rangle \times \begin{cases} \rho_{\rm in} & \text{modes out to } \tau_h + \text{the buffer,} \\
\rho_{\rm ext} & \text{the modes beyond (the extension),} \end{cases}$$
with $\rho_{\rm in}$ = `DATA_PRIOR_RIDGE` and $\rho_{\rm ext}$ = `DATA_PRIOR_EXTENSION_RIDGE`.
* **Why the ridges:** without them, a combination of modes localized inside a wide gap, with a fraction $e \ll 1$ of its weighted energy
  on observed cells, would be fit to the noise with its variance amplified by $1/e$. With them, no coefficient carries more noise
  variance than $\sigma^2 / (4\rho)$, and the fit is the unique solution of a well-posed problem, the same wherever the solver stops.
* **The split:** eigenvalue-truncated DPSS spans do not nest, so the full span is rotated into the part along the inner modes and the
  rest: with the QR decomposition $F_{\rm full}^\dagger F_{\rm in} = Q\,\mathcal{R}$, the columns of $F_{\rm full}\, Q$ are the inner
  modes (first) and the extension.
* **Autocorrelations:** bright enough that the data pin down every mode of the plain fit (no ridge, which would only bias it).
* **When it is fit:** only where a cell the fill uses lacks a usable model (see the main loop).

In [ ]:
def data_only_prior(data, weights, tslices, bands):
    '''The prior where the sky model is missing or flagged, and for autocorrelations: a 2D DPSS fit of the data (with these weights)
    over the sky's fringe rates, out to the baseline's horizon plus DATA_PRIOR_DELAY_BUFFER in delay, and for a cross-correlation at
    least DATA_PRIOR_MIN_DELAY, to carry the mutual coupling. For a cross-correlation, ridges (relative to the mean weight) pull the
    combinations of modes that the data leave unconstrained (deep in wide gaps) back to zero, so the fit is the solution of a
    well-posed problem rather than wherever the solver stops: a light one (DATA_PRIOR_RIDGE) on the modes out to the horizon plus the
    buffer, and a heavier one (DATA_PRIOR_EXTENSION_RIDGE) on the modes beyond. An autocorrelation is bright enough that the data pin
    down every mode of its plain fit, which needs no ridge (any would bias it). Returns the fit (np.nan outside the bands and time
    slices fit) and a short description of it, for the history.'''
    antpair = data.antpairs().pop()
    horizon_ns = np.linalg.norm(data.antpos[antpair[1]] - data.antpos[antpair[0]]) / constants.c * 1e9
    inner = horizon_ns + DATA_PRIOR_DELAY_BUFFER
    if antpair[0] == antpair[1]:
        return fit_2D_DPSS(data, weights, inner, tslices, bands, atol=CG_TOL, btol=CG_TOL), f'out to {inner:.0f} ns'
    reach = max(inner, DATA_PRIOR_MIN_DELAY)
    fit = copy.deepcopy(data)
    for bl in data:
        fit[bl] = np.full_like(data[bl], np.nan)
        for tslice, band in zip(tslices[bl], bands[bl]):
            if (tslice is None) or (band is None) or np.all(weights[bl][tslice, band] == 0):
                continue
            wgts = weights[bl][tslice, band]
            fr_center, fr_hw = FR_CENTER_AND_HW_CACHE[(bl[0:2], tslice, band)]
            time_filters, _ = dpss_operator((data.times[tslice] - data.times[tslice][0]) * 3600 * 24, [fr_center], [fr_hw], eigenval_cutoff=[EIGENVAL_CUTOFF])
            full_filters, _ = dpss_operator(data.freqs[band], [0.0], [reach / 1e9], eigenval_cutoff=[EIGENVAL_CUTOFF])
            inner_filters, _ = dpss_operator(data.freqs[band], [0.0], [inner / 1e9], eigenval_cutoff=[EIGENVAL_CUTOFF])
            # rotate the full span into the part along the inner modes (its first columns) and the rest (DPSS spans truncated by
            # eigenvalue do not nest, so the inner modes themselves are not quite inside it); the rest is empty when reach == inner
            freq_filters = full_filters @ np.linalg.qr(full_filters.conj().T @ inner_filters, mode='complete')[0]
            nt, nf = time_filters.shape[1], freq_filters.shape[1]
            sqrt_w = np.sqrt(wgts)
            ridges = np.where(np.arange(nf) < inner_filters.shape[1], DATA_PRIOR_RIDGE, DATA_PRIOR_EXTENSION_RIDGE)
            root_lam = np.sqrt(ridges * np.mean(wgts[wgts > 0]))   # per frequency mode, for every time mode

            # x is the (time mode x frequency mode) coefficient matrix, flattened; the weighted model on every cell of the block is
            # followed by the ridge's rows, one per coefficient
            def matvec(x):
                x = x.reshape(nt, nf)
                return np.concatenate([(sqrt_w * (time_filters @ x @ freq_filters.T)).ravel(), (root_lam * x).ravel()])

            # LSQR also needs the adjoint: the same products, transposed and conjugated
            def rmatvec(y):
                y_cells = sqrt_w * y[:wgts.size].reshape(wgts.shape)
                return (time_filters.T.conj() @ y_cells @ freq_filters.conj() + root_lam * y[wgts.size:].reshape(nt, nf)).ravel()

            A = spla.LinearOperator((wgts.size + nt * nf, nt * nf), matvec=matvec, rmatvec=rmatvec, dtype=complex)
            target = np.concatenate([(sqrt_w * np.where(wgts > 0, data[bl][tslice, band], 0)).ravel(), np.zeros(nt * nf, complex)])
            x, istop = spla.lsqr(A, target, atol=CG_TOL, btol=CG_TOL, iter_lim=CG_ITER_LIM)[:2]
            if istop == 7:   # LSQR's stop code 7: it reached iter_lim without meeting atol/btol
                print(f'\tWARNING: the data-only fit of {bl} at {data.freqs[band][0] / 1e6:.0f}-{data.freqs[band][-1] / 1e6:.0f} MHz stopped '
                      f'at the iteration cap ({CG_ITER_LIM}) without converging.')
            fit[bl][tslice, band] = time_filters @ x.reshape(nt, nf) @ freq_filters.T
    return fit, f'out to {reach:.0f} ns, ridge {DATA_PRIOR_RIDGE:g}' + (f' to {inner:.0f} ns and {DATA_PRIOR_EXTENSION_RIDGE:g} beyond' if reach > inner else '')

### Blending the model into the refit

Where the sky model covers only part of the night, a first fill about the mixed prior (the adjusted model where it is usable, the
data-only fit elsewhere) is refit by the data-only fit, with the filled cells counted as data. That refit, $f_{\rm re}$, is continuous
across the edge of the model's coverage. The final prior is
$$p(t, \nu) = \begin{cases} f_{\rm re}(t, \nu) & \text{in uncovered integrations, and where the model is flagged,} \\
a(t)\, \hat m(t, \nu) + \big[1 - a(t)\big]\, f_{\rm re}(t, \nu) & \text{in covered integrations,} \end{cases}
\qquad a(t) = \min\!\left(1, \frac{\Delta t_{\rm edge}(t)}{T_{\rm blend}}\right),$$
where $\Delta t_{\rm edge}$ is the time to the nearest uncovered integration (separately in each band) and $T_{\rm blend}$ =
`WIENER_TAPER_HOURS`, the covariance's timescale: the prior moves smoothly from the refit at the edge to the adjusted model an hour into
the coverage.

In [ ]:
def blend_into_refit(model, refit, usable, tslices, bands):
    '''The final prior where a sky model covers only part of the night: the refit (a data-only fit) wherever the model is not
    usable, and the model elsewhere, blended linearly into the refit over the last WIENER_TAPER_HOURS of coverage before each edge
    (separately in each band), so that the prior does not step where the model's coverage ends.'''
    prior = {}
    for bl in model:
        prior[bl] = np.where(usable[bl], model[bl], refit[bl])
        for tslice, band in zip(tslices[bl], bands[bl]):
            if (tslice is None) or (band is None):
                continue
            covered_rows = np.any(usable[bl][:, band], axis=1)
            if np.all(covered_rows) or not np.any(covered_rows):
                continue
            # integrations from the nearest uncovered one, as a fraction of the blend: 0 at the edge, 1 a taper width into the coverage
            dist = np.min(np.abs(np.arange(len(covered_rows))[:, None] - np.flatnonzero(~covered_rows)[None, :]), axis=1)
            a = np.clip(dist * dt / 3600 / WIENER_TAPER_HOURS, 0, 1)[:, None]
            prior[bl][:, band] = np.where(usable[bl][:, band], a * model[bl][:, band] + (1 - a) * refit[bl][:, band], refit[bl][:, band])
    return prior

### The residual's covariance

The fill works on the residual about the prior in autocorrelation units,
$$r(t, \nu) = \frac{V(t, \nu) - p(t, \nu)}{\sqrt{|A_i(t, \nu)\, A_j(t, \nu)|}},$$
with $A_i$ and $A_j$ from the smooth 2D model of the autocorrelations. That flattens the sky's and the noise's steep spectra, and the
noise variance becomes $\sigma^2 = \epsilon / (n_{\rm eff}\, \Delta t\, \Delta\nu)$, where $\epsilon$ = `NOISE_EXCESS` is the
correlator's excess over the radiometer equation (from `hera_cal.noise.correlator_noise_covariance` and `CORRELATOR`). The
correlator also correlates the noise of adjacent channels by $\rho$ = `NOISE_CHANNEL_CORRELATION`. The residual's covariance is taken to be stationary in time
and frequency, $C(\Delta t, \Delta\nu) = \big\langle r(t + \Delta t, \nu + \Delta\nu)\, r^*(t, \nu) \big\rangle$, and estimated from the
observed cells (mask $M$) by the Wiener–Khinchin theorem on a zero-padded $(2N_t, 2N_\nu)$ grid, so that lags do not wrap around:
$$\hat C(\Delta t, \Delta\nu) = \frac{\mathcal{F}^{-1}\big[\,|\mathcal{F}[M r]|^2\,\big]}{\mathcal{F}^{-1}\big[\,|\mathcal{F}[M]|^2\,\big]}
- \langle \sigma^2 \rangle_M\, \delta_{\Delta t, 0}\, \big(\delta_{\Delta \nu, 0} + \rho\, \delta_{|\Delta \nu|, \Delta\nu_1}\big),$$
the mean lag product over every observed pair (lags with fewer than 100 pairs dropped), with the noise removed: its variance at zero lag,
and $\rho$ times that between adjacent channels ($\Delta\nu_1$ is one channel). Its 2D Fourier transform, the power spectrum in fringe rate and delay, is
$$\tilde C_{\rm raw}(f, \tau) = \mathrm{Re}\; \mathcal{F}\big[\hat C(\Delta t, \Delta\nu)\, W_t(\Delta t)\, W_\nu(\Delta\nu)\big],$$
with Blackman–Harris lag tapers $W$ whose equivalent widths (area over peak) are `WIENER_TAPER_HOURS` and `WIENER_TAPER_MHZ`. They set
$\tilde C$'s resolution (their reciprocals) and end the covariance at their zeros (the equivalent width over 0.7175). What the noise model
gets wrong leaves a flat floor $\tilde C_0$, the mean of $\tilde C_{\rm raw}$ beyond the sky's fringe rates and beyond 2 µs. It is subtracted, and
only the modes significantly above it are kept:
$$\tilde C(f, \tau) = \begin{cases} \tilde C_{\rm raw} - \tilde C_0 & \text{if } \tilde C_{\rm raw} - \tilde C_0 > k\, \sigma_{\tilde C}, \\ 0 & \text{otherwise,} \end{cases}$$
with $\sigma_{\tilde C}$ = 1.4826 × the median absolute deviation of $\tilde C_{\rm raw}$ and $k$ = `WIENER_THRESHOLD_SIGMA`. The covariance the
fill uses is its inverse transform, $C = \mathcal{F}^{-1}[\tilde C]$.

In [ ]:
FFT_WORKERS = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else os.cpu_count()   # the CPUs this job may use


def lag_taper(n, width):
    '''A Blackman-Harris lag taper on a 2n FFT grid (fft order) whose equivalent width (area over peak) is `width` samples. It
    reaches zero at |lag| = K samples (never past the block), and returns the taper and K.'''
    # a window's equivalent width is a fixed fraction of its zero (0.7175 for Blackman-Harris): read it off a long copy
    K = min(int(np.round(width / (np.sum(gen_window('bh', 20001)) / 10000))), n - 1)
    taper = np.zeros(2 * n)
    taper[np.r_[-K:K + 1]] = gen_window('bh', 2 * K + 1)   # lags -K..K: FFT order puts the negative ones at the end
    return taper, K


def residual_power_spectrum(r, obs, noise_var, fr_center, fr_hw):
    '''The residual's 2D power spectrum P(fringe rate, delay) on the zero-padded (2 Nt, 2 Nf) grid, assuming it is stationary in time
    and frequency: masked lag products averaged over every observed pair (lags with fewer than 100 pairs dropped), the noise
    removed (its variance at zero lag and its covariance between adjacent channels), and Blackman-Harris lag tapers with equivalent widths of WIENER_TAPER_HOURS and WIENER_TAPER_MHZ (P's
    resolution is their reciprocal). The flat floor left by the noise model's error (the mean beyond the sky fringe rates and
    beyond 2 us) is subtracted, and modes below WIENER_THRESHOLD_SIGMA times the estimate's scatter (from its median absolute
    deviation) are dropped. Returns P and what regenerates it: the tapered lag function's non-negative time lags and the scalars.'''
    nt, nf = r.shape
    taper_t, Kt = lag_taper(nt, WIENER_TAPER_HOURS * 3600 / dt)
    taper_f, Kf = lag_taper(nf, WIENER_TAPER_MHZ * 1e6 / df)
    # Wiener-Khinchin on the zero-padded grid (so lags don't wrap around): |FFT2(masked r)|^2 transforms back to the sum over
    # observed pairs of r(t + lag_t, nu + lag_nu) r*(t, nu), and |FFT2(mask)|^2 to the number of those pairs
    R = sfft.fft2(np.where(obs, r, 0), s=(2 * nt, 2 * nf), workers=FFT_WORKERS)
    M = sfft.fft2(obs.astype(float), s=(2 * nt, 2 * nf), workers=FFT_WORKERS)
    lags = sfft.ifft2(np.abs(R)**2, workers=FFT_WORKERS)
    counts = sfft.ifft2(np.abs(M)**2, workers=FFT_WORKERS).real
    c = np.where(counts > 100, lags / np.maximum(counts, 1), 0)   # the mean lag product: the covariance at each lag
    # the noise adds its variance at zero lag and, through the correlator, a small covariance between adjacent channels at the same
    # time: remove both, leaving the signal's covariance
    c[0, 0] -= np.mean(noise_var[obs])
    c[0, [1, -1]] -= NOISE_CHANNEL_CORRELATION * np.mean(noise_var[obs])
    # the lag tapers set P's resolution (the reciprocals of their equivalent widths) and end the covariance at their zeros. The
    # tapered lag function is Hermitian, c(-lag) = c(lag)*, so its 2D FFT, the power spectrum in fringe rate and delay, is real
    c *= taper_t[:, None] * taper_f[None, :]
    P_raw = np.real(sfft.fft2(c, workers=FFT_WORKERS))
    sigma_P = 1.4826 * np.median(np.abs(P_raw - np.median(P_raw)))   # the estimate's robust scatter (1.4826 MAD = sigma for a Gaussian)
    # beyond the sky fringe rates and beyond 2 us there should be no signal, so P_raw there is a flat floor from the noise model's
    # error: subtract it, then keep only the modes significantly above it
    beyond = (np.abs(np.fft.fftfreq(2 * nt, dt)) > np.abs(fr_center) + fr_hw)[:, None] & (np.abs(np.fft.fftfreq(2 * nf, df)) > 2e-6)[None, :]
    floor = np.mean(P_raw[np.broadcast_to(beyond, P_raw.shape)])
    P = np.where(P_raw - floor > WIENER_THRESHOLD_SIGMA * sigma_P, P_raw - floor, 0)
    # what regenerates P: the tapered lag function vanishes beyond the zeros, and c(-lag_t, -lag_nu) = c(lag_t, lag_nu)*, so time
    # lags 0..Kt with frequency lags -Kf..Kf hold all of it
    half = c[:Kt + 1][:, np.concatenate([np.arange(0, Kf + 1), np.arange(2 * nf - Kf, 2 * nf)])].astype(np.complex64)
    return P, dict(half_lag_function=half, nt=nt, nf=nf, Kt=Kt, Kf=Kf, sigma_P=sigma_P, noise_floor=floor, mean_noise=np.mean(noise_var[obs]))

### The 2D Wiener fill

Given the covariance, the fill is the conditional mean of a Gaussian process regression (GPR) of $r$ with signal covariance $C$ and
noise covariance $N$, which has $\sigma^2$ on its diagonal and $\rho\, \sigma_k \sigma_{k+1}$ between adjacent channels at the same time,
given $r$ at the observed cells $\mathcal{O}$, on every cell of the block:
$$\hat r = C_{\cdot\, \mathcal{O}}\, \big(C_{\mathcal{O}\mathcal{O}} + N_{\mathcal{O}}\big)^{-1} r_{\mathcal{O}}.$$
* **The solve:** in data space, by conjugate gradients for $\alpha$ in $\big(C_{\mathcal{OO}} + N_{\mathcal{O}}\big)\, \alpha = r_{\mathcal{O}}$,
  then $\hat r = C_{\cdot\, \mathcal{O}}\, \alpha$.
* **Applying $C$:** as a convolution on the zero-padded grid, $C x = \mathcal{F}^{-1}\big[\tilde C\, \mathcal{F}[x]\big]$, so no covariance matrix
  is ever formed.
* **The preconditioner:** it inverts $C + N$ as if every cell were observed with the mean noise, which is diagonal in Fourier space.
  There $N$'s power is $\tilde N(\tau) = \langle \sigma^2 \rangle \big[1 + 2\rho \cos(2\pi \tau \Delta\nu_1)\big]$, so the preconditioner is
  $y \mapsto \mathcal{F}^{-1}\big[\mathcal{F}[y] / (\tilde C + \tilde N)\big]$.
* **Convergence:** since $N_{\mathcal{O}}$ is positive definite, the system is always well posed; CG stops at a relative residual of
  `CG_TOL`, within `CG_ITER_LIM` iterations.

In [ ]:
def wiener_fill_block(r, obs, noise_var, P):
    '''The 2D Wiener fill (the conditional mean of a GPR with the stationary covariance behind P) of r everywhere in the block,
    given r at the observed cells: conjugate gradients on (C_oo + N_o) alpha = r_o in data space, applying C as a zero-padded FFT
    convolution and preconditioning by the full-grid inverse 1 / (P + the mean noise's power); the fill is C_(all, o) alpha. The noise
    covariance N has noise_var on its diagonal and the correlator's covariance between adjacent channels next to it.'''
    nt, nf = r.shape
    shape = (2 * nt, 2 * nf)
    # C applied to a field on the block: multiplying by P in Fourier space convolves with the covariance, and the zero padding keeps
    # that convolution from wrapping around the block's edges
    conv = lambda x: sfft.ifft2(P * sfft.fft2(x, s=shape, workers=FFT_WORKERS), workers=FFT_WORKERS)[:nt, :nf]
    mean_noise = np.mean(noise_var[obs])
    sigma = np.sqrt(np.where(obs, noise_var, 0))   # zero off the observed cells, so only observed neighbors share noise

    def embed(v):   # a vector over the observed cells, put back on the block with zeros elsewhere
        x = np.zeros((nt, nf), complex)
        x[obs] = v
        return x

    def noise_cov(v):   # N_o applied to a vector over the observed cells: sigma_k^2 v_k + rho sigma_k sigma_{k+-1} v_{k+-1}
        x = sigma * embed(v)
        neighbors = np.zeros_like(x)
        neighbors[:, 1:] += x[:, :-1]
        neighbors[:, :-1] += x[:, 1:]
        return (sigma * (x + NOISE_CHANNEL_CORRELATION * neighbors))[obs]

    # A = C_oo + N_o: the covariance of the observed residual (signal plus its own noise), acting on vectors over the observed cells
    A = spla.LinearOperator((obs.sum(),) * 2, matvec=lambda v: conv(embed(v))[obs] + noise_cov(v), dtype=complex)
    # the preconditioner inverts C + N as if every cell were observed with the mean noise, which is diagonal in Fourier space: divide by
    # P plus the noise's power, which the correlation between adjacent channels makes a cosine in delay
    noise_power = mean_noise * (1 + 2 * NOISE_CHANNEL_CORRELATION * np.cos(2 * np.pi * np.fft.fftfreq(2 * nf)))[None, :]
    M = spla.LinearOperator((obs.sum(),) * 2, dtype=complex,
                            matvec=lambda v: sfft.ifft2(sfft.fft2(embed(v), s=shape, workers=FFT_WORKERS) / (P + noise_power), workers=FFT_WORKERS)[:nt, :nf][obs])
    # alpha = (C_oo + N_o)^-1 r_o; the fill on every cell of the block is then C_(all, o) alpha, the conditional mean
    n_iter = [0]
    alpha, info = spla.cg(A, r[obs], M=M, rtol=CG_TOL, maxiter=CG_ITER_LIM, callback=lambda xk: n_iter.__setitem__(0, n_iter[0] + 1))
    return conv(embed(alpha)), dict(iterations=n_iter[0], converged=(info == 0))

### Inpainting a baseline, and the covariance file

For each polarization and band, `wiener_inpaint` forms the residual about the prior, learns $\tilde C$ from the observed cells (or, where the
model covers only part of the night, from the covered integrations), fills it, and returns to visibility units at the cells being
inpainted:
$$V_{\rm inpainted}(t, \nu) = p(t, \nu) + \sqrt{|A_i A_j|}\; \hat r(t, \nu).$$
* Flagged cells that are not inpainted keep the prior.
* A co-polarized autocorrelation is made real.
* Each polarization and band's covariance is written out by `write_inpaint_covariance`, for assessing the noise of the inpainted and
  LST-stacked data: the tapered lag function's non-negative time lags (the rest follows from
  $C(-\Delta t, -\Delta\nu) = C^*(\Delta t, \Delta\nu)$), with the scalars that regenerate the thresholded $\tilde C$ exactly ($\sigma_{\tilde C}$,
  $\tilde C_0$, $\langle \sigma^2 \rangle$, $k$, and the tapers).

In [ ]:
def wiener_inpaint(data, flags, prior, weights, ip_nsamples, where_inpainted, tslices, bands, prior_note, covariance_cells=None):
    '''Inpaint every baseline's where_inpainted cells by the 2D Wiener fill of the residual about the prior, in autocorrelation units
    (the residual over sqrt|A_i A_j| from the smooth model of the autos), with its noise variance predicted from the autos and the
    effective nsamples, 1 / (nsamples dt df). One covariance per polarization and band, learned from the observed cells in
    covariance_cells (by default all of them). Flagged cells that are not inpainted get the prior (np.nan priors count as 0).
    Returns the inpainted data and, per polarization and band, what regenerates the covariance.'''
    ip_data = copy.deepcopy(data)
    covariances = {}
    for bl in data:
        # work in autocorrelation units: dividing by sqrt|A_i A_j| (from the smooth auto model) flattens the sky's and the noise's
        # steep spectra, so a covariance stationary in frequency fits better, and the noise variance becomes 1 / (nsamples dt df)
        ant1, ant2 = utils.split_bl(bl)
        auto_bl_1 = [k for k in auto_fit if k[2] == utils.join_pol(ant1[1], ant1[1])][0]
        auto_bl_2 = [k for k in auto_fit if k[2] == utils.join_pol(ant2[1], ant2[1])][0]
        amp = np.sqrt(np.abs(auto_fit[auto_bl_1] * auto_fit[auto_bl_2]))
        with np.errstate(divide='ignore', invalid='ignore'):   # nsamples can be 0 in fully flagged integrations
            noise_var = NOISE_EXCESS / (ip_nsamples[bl] * dt * df)   # the radiometer equation, with the correlator's excess
        prior_here = np.where(np.isfinite(prior[bl]), prior[bl], 0)
        ip_data[bl] = np.where(flags[bl], prior_here, data[bl])
        for tslice, band in zip(tslices[bl], bands[bl]):
            if (tslice is None) or (band is None):
                continue
            sl = (tslice, band)
            # the cells the fill conditions on: unflagged, with weight, and with a usable noise level and auto amplitude
            obs = ~flags[bl][sl] & (weights[bl][sl] > 0) & np.isfinite(noise_var[sl]) & (amp[sl] > 0)
            if not np.any(obs) or not np.any(where_inpainted[bl][sl]):
                continue
            r = np.where(obs, (data[bl][sl] - prior_here[sl]) / np.where(obs, amp[sl], 1), 0)
            nv = np.where(obs, noise_var[sl], 0)
            fr_center, fr_hw = FR_CENTER_AND_HW_CACHE[(bl[0:2], tslice, band)]
            P, cov = residual_power_spectrum(r, obs if covariance_cells is None else obs & covariance_cells[bl][sl], nv, fr_center, fr_hw)
            fill, info = wiener_fill_block(r, obs, nv, P)
            # back to visibility units at the cells being inpainted: the prior plus sqrt|A_i A_j| times the filled residual
            ip_data[bl][sl] = np.where(where_inpainted[bl][sl], prior_here[sl] + amp[sl] * fill, ip_data[bl][sl])
            band_name = f'{data.freqs[band][0] / 1e6:.0f}-{data.freqs[band][-1] / 1e6:.0f} MHz'
            cov.update(tslice=[tslice.start, tslice.stop], band=[band.start, band.stop], jd_range=data.times[tslice][[0, -1]],
                       freq_range_hz=data.freqs[band][[0, -1]], modes_kept=np.mean(P > 0), cg_iterations=info['iterations'],
                       cg_converged=info['converged'], prior=prior_note)
            covariances[f'{bl[2]}/{band_name}'] = cov
            print(f'\t{bl[2]} {band_name}: 2D fill of {np.sum(where_inpainted[bl][sl])} cells, {np.mean(P > 0):.2%} of the covariance\'s modes kept, '
                  f'CG {info["iterations"]} iterations' + ('' if info['converged'] else f' (NOT converged in {CG_ITER_LIM}: keeping the last iterate)'))
        if ant1 == ant2:   # a co-polarized autocorrelation is real
            ip_data[bl] = ip_data[bl].real.astype(ip_data[bl].dtype)
    return ip_data, covariances


def write_inpaint_covariance(outfile, covariances, antpair, history):
    '''Write each polarization and band's covariance as the tapered lag function's non-negative time lags (the rest follows by
    Hermitian symmetry), with the scalars that regenerate the thresholded P exactly, and the inpainted data's history.'''
    with h5py.File(outfile, 'w') as h:
        h.attrs.update(dict(antpair=list(antpair), dt_s=dt, df_hz=df, taper='blackman-harris', equivalent_width_hours=WIENER_TAPER_HOURS,
                            equivalent_width_mhz=WIENER_TAPER_MHZ, threshold_sigma=WIENER_THRESHOLD_SIGMA,
                            noise_excess=NOISE_EXCESS, noise_channel_correlation=NOISE_CHANNEL_CORRELATION,
                            units='autocorrelation units (residual / sqrt|A_i A_j|)',
                            regenerate=('embed half_lag_function in a (2 nt, 2 nf) grid at time lags 0..Kt and frequency lags 0..Kf and -Kf..-1, '
                                        'fill the negative time lags by Hermitian symmetry, take the real part of its fft2, subtract noise_floor, '
                                        'and zero every mode below threshold_sigma * sigma_P')))
        h.create_dataset('history', data=history)   # a dataset, not an attribute: the history outgrows HDF5's 64 kB limit on attributes
        for key, cov in covariances.items():
            d = h.create_dataset(key, data=cov['half_lag_function'])
            d.attrs.update({k: v for k, v in cov.items() if k != 'half_lag_function'})

## Generate smooth model of autos for noise modeling

In [ ]:
# the night's averaged autocorrelations, for noise modeling
all_outfiles = [outfile for outfiles in corner_turn_map['files_to_outfiles_map'].values() for outfile in outfiles]
for outfile in all_outfiles:
    match = re.search(r'\.(\d+)_(\d+)\.', os.path.basename(outfile))
    if match and match.group(1) == match.group(2):
        print(f'Loading {outfile} for autocorrelations to use for noise modeling.')
        hd_autos = io.HERAData(outfile)
        autos, auto_flags, auto_nsamples = hd_autos.read(polarizations=['ee', 'nn'])
        dt = np.median(np.diff(hd_autos.times)) * 24 * 3600
        df = np.median(np.diff(hd_autos.freqs))
        for bl in auto_flags:
            auto_flags[bl] |= prior_flags
        break

In [ ]:
weights = {}
tslices = {}
bands = {}
for bl in autos:
    noise = 2 * np.abs(autos[bl]) / (auto_nsamples[bl] * dt * df)**.5
    weights[bl] = np.where(auto_flags[bl], 0, noise**-2)
    weights[bl] /= np.mean(weights[bl][weights[bl] > 0])
    tslices[bl], bands[bl] = flag_utils.get_minimal_slices(weights[bl] == 0, freqs=autos.freqs, freq_cuts=[BAND_SPLIT_FREQ * 1e6])
    for tslice, band in zip(tslices[bl], bands[bl]):
        cache_fr_center_and_hw(hd_autos, bl[0:2], tslice, band)

In [ ]:
auto_fit = fit_2D_DPSS(autos, weights, AUTO_INPAINT_DELAY, tslices, bands, atol=CG_TOL, btol=CG_TOL)

In [ ]:
# remove unused objects to save memory
del hd_autos, autos, auto_nsamples

## Main loop: the 2D Wiener fill about the prior

For each of this job's single-baseline files:
* apply the final flags;
* subtract the polarized source models;
* build the prior (below);
* fill every flagged cell by the 2D Wiener fill;
* write
    * the inpainted data,
    * the filled effective `nsamples`,
    * the `where_inpainted` sidecar,
    * and the fill's covariances.

The prior:
* **Without a sky model, and for autocorrelations:** the data-only 2D DPSS fit.
* **With a sky model:** the model is *usable* wherever it is unflagged, and an integration counts as *covered* (separately above and
  below `BAND_SPLIT_FREQ`) where it has any usable cell. The model is fit to the night, and the prior is that adjusted model where
  usable and the data-only fit elsewhere (fit only if some cell the fill uses lacks a usable model).
    * **The model covers every integration to inpaint:** one fill about that prior finishes the job.
    * **It covers only some:**
        * the covariance is learned from the covered integrations (when they span at least twice the lag taper's reach in time);
        * a first fill about the mixed prior is refit by the data-only fit, with that fill counted as data;
        * the final fill uses the model blended into that refit over the last `WIENER_TAPER_HOURS` before each edge of its coverage.

In [ ]:
adjustment_figs = []
waterfall_figs = []
covariance_figs = []

for single_bl_file in single_bl_files:

    # load data
    if not os.path.exists(single_bl_file):
        print(f'{single_bl_file} does not exist (likely skipped upstream). Skipping...')
        continue
    print(f"Now loading {single_bl_file}")
    hd = io.HERAData(single_bl_file)
    data, flags, nsamples = hd.read()
    dt = np.median(np.diff(hd.times)) * 24 * 3600
    df = np.median(np.diff(hd.freqs))
    antpair = data.antpairs().pop()
    is_auto = (antpair[0] == antpair[1])

    # apply the night's final flags
    for bl in flags:
        flags[bl] |= prior_flags

    if np.all([flags[bl] for bl in flags]):
        print('\tThis baseline is entirely flagged. Skipping...')
        continue

    if SUBTRACT_POLARIZED_SOURCE and len(model_specs) > 0 and not is_auto:
        print("\tSubtracting off a polarized source model from the visibilities...")
        subtract_polarized_models(data, flags, model_specs, baseline=antpair)

    # get minimal slices, and the fringe-rate ranges the 2D fits use
    tslices = {}
    bands = {}
    for bl in data:
        tslices[bl], bands[bl] = flag_utils.get_minimal_slices(flags[bl], freqs=data.freqs, freq_cuts=[BAND_SPLIT_FREQ * 1e6])
        for tslice, band in zip(tslices[bl], bands[bl]):
            cache_fr_center_and_hw(hd, bl[0:2], tslice, band)

    # fill in effective nsamples at flagged cells (used for weighting, and written out where inpainted)
    ip_nsamples = get_ip_nsamples(nsamples, flags, tslices, bands)

    # get weights from smooth auto model
    weights_before_ip, weights_after_ip = get_weights_for_inpainting(data, flags, tslices, bands, ip_nsamples, auto_fit, auto_flags)

    # set up ip_flags and where_inpainted
    ip_flags = copy.deepcopy(flags)
    where_inpainted = copy.deepcopy(flags)
    for bl in ip_flags:
        ip_flags[bl][:, :] = True
        where_inpainted[bl][:, :] = False
        for tslice, band in zip(tslices[bl], bands[bl]):
            if (tslice is None) or (band is None):
                continue
            ip_flags[bl][tslice, band] = np.all(weights_before_ip[bl][tslice, band] == 0, axis=1, keepdims=True)
            where_inpainted[bl][tslice, band] = flags[bl][tslice, band] & (~ip_flags[bl][tslice, band])

    # the prior: the sky model of this redundant group (matched to the night's LSTs and fit to it) where it is usable, and a 2D
    # DPSS fit of the night's data elsewhere (only fit if some cell the fill uses has no usable model)
    sky_model_file = None if is_auto else find_sky_model_file(antpair)   # a stack's autos fit one night's poorly: autos get the data-only prior
    sky_model = None
    if sky_model_file is not None:
        print(f'\tLoading sky model from {sky_model_file}')
        sky_model, sky_model_flags = load_sky_model(hd, data, sky_model_file)
    if sky_model is not None and SUBTRACT_POLARIZED_SOURCE and SUBTRACT_POLARIZED_SOURCE_FROM_SKY_MODEL and len(model_specs) > 0 and not is_auto:
        print("\tSubtracting the same polarized source model from the sky model...")
        subtract_polarized_models(sky_model, sky_model_flags, model_specs, baseline=antpair)

    if sky_model is None:
        print(f'\t{"An autocorrelation" if is_auto else "No sky model for this baseline"}: the prior is a 2D DPSS fit of the data.')
        data_prior, data_prior_desc = data_only_prior(data, weights_before_ip, tslices, bands)
        prior_note = f'a 2D DPSS fit of the data ({data_prior_desc}), as the prior of a 2D Wiener fill'
        ip_data, covariances = wiener_inpaint(data, flags, data_prior, weights_before_ip, ip_nsamples, where_inpainted, tslices, bands, prior_note)
    else:
        # the model is usable wherever it is unflagged; an integration is covered, separately in each band, where it has any usable cell
        usable = {bl: ~sky_model_flags[bl] for bl in data}
        covered = {bl: np.zeros_like(flags[bl]) for bl in data}
        for bl in data:
            for tslice, band in zip(tslices[bl], bands[bl]):
                if (tslice is None) or (band is None):
                    continue
                covered[bl][:, band] = np.any(usable[bl][:, band], axis=1, keepdims=True)
        n_stand_in = sum(np.sum(~flags[bl] & covered[bl] & ~usable[bl]) for bl in data)
        if n_stand_in > 0:
            print(f'\tThe model is flagged at {n_stand_in} unflagged cells of integrations it covers: the data-only fit stands in there.')
        n_covered = sum(np.sum(where_inpainted[bl] & covered[bl]) for bl in data)
        n_uncovered = sum(np.sum(where_inpainted[bl] & ~covered[bl]) for bl in data)
        sky_model, G, R = adjust_sky_model_to_data(data, flags, sky_model, sky_model_flags, weights_before_ip, tslices, bands)
        if any(np.any(np.isfinite(G[bl]) & ~sky_model_flags[bl]) for bl in data):   # Figure 1, made now so that G and R can go
            adjustment_figs.append(sky_model_adjustment_figure(data, G, R, sky_model_flags, ip_nsamples, close=True))
        del G, R
        mixed, data_prior_desc = sky_model, None
        if any(np.any(~usable[bl] & ~ip_flags[bl]) for bl in data):   # a cell the fill uses (observed or inpainted) has no usable model
            data_prior, data_prior_desc = data_only_prior(data, weights_before_ip, tslices, bands)
            mixed = {bl: np.where(usable[bl], sky_model[bl], data_prior[bl]) for bl in data}
        model_note = f'sky model {sky_model_file}, fit to the night by a smooth gain and an additive correction (ridge {SKY_MODEL_ADJUST_RIDGE:g})'
        if not any(np.any(~ip_flags[bl] & ~covered[bl]) for bl in data):   # the model covers every integration with data
            prior_note = f'{model_note}, as the prior of a 2D Wiener fill'
            if data_prior_desc is not None:   # the model is flagged at some cells the fill uses
                prior_note += f', with a 2D DPSS fit of the data ({data_prior_desc}) standing in where the model is flagged'
            ip_data, covariances = wiener_inpaint(data, flags, mixed, weights_before_ip, ip_nsamples, where_inpainted, tslices, bands, prior_note)
        else:
            # the model covers only part of the night: learn the covariance where it covers (if that spans at least twice the lag
            # taper's reach in time), fill about the mixed prior, refit the data-only prior with that fill counted as data, and fill
            # again about the model blended into the refit before each edge of its coverage
            print(f'\tThe model covers {n_covered} of the {n_covered + n_uncovered} cells to inpaint.')
            K = lag_taper(len(data.times), WIENER_TAPER_HOURS * 3600 / dt)[1]
            covariance_cells = {bl: np.ones_like(flags[bl]) for bl in data}
            for bl in data:
                for tslice, band in zip(tslices[bl], bands[bl]):
                    if (tslice is not None) and (band is not None) and np.sum(covered[bl][:, band][:, 0]) >= 2 * K:
                        covariance_cells[bl][:, band] = covered[bl][:, band]
            first_fill, _ = wiener_inpaint(data, flags, mixed, weights_before_ip, ip_nsamples, where_inpainted, tslices, bands, 'first pass',
                                           covariance_cells=covariance_cells)
            refit, refit_desc = data_only_prior(first_fill, {bl: np.where(~flags[bl] | where_inpainted[bl], weights_after_ip[bl], 0) for bl in data},
                                                tslices, bands)
            prior_note = (f'{model_note} for {n_covered} of {n_covered + n_uncovered} inpainted cells, blended over the last '
                          f'{WIENER_TAPER_HOURS:g} h of its coverage into a 2D DPSS refit of the data ({refit_desc}) and a first fill, '
                          'as the prior of a 2D Wiener fill')
            ip_data, covariances = wiener_inpaint(data, flags, blend_into_refit(sky_model, refit, usable, tslices, bands), weights_before_ip,
                                                  ip_nsamples, where_inpainted, tslices, bands, prior_note, covariance_cells=covariance_cells)
    print(f'\tPrior: {prior_note}.')

    # the filled effective nsamples at the inpainted cells, for LST stacking's weights; other cells keep their own
    ip_nsamples_out = copy.deepcopy(nsamples)
    for bl in ip_nsamples_out:
        ip_nsamples_out[bl] = np.where(where_inpainted[bl], ip_nsamples[bl], nsamples[bl])

    # perform FR=0 filter, if desired
    if FR0_FILTER and not is_auto:
        fr0_filt_ip_data = copy.deepcopy(ip_data)
        for bl in fr0_filt_ip_data:
            for tslice, band in zip(tslices[bl], bands[bl]):
                if (tslice is None) or (band is None):
                    continue
                wgts_here = np.where(ip_flags[bl], 0, weights_after_ip[bl])[tslice, band]
                d_mdl, _, info = fourier_filter(data.times[tslice] * 24 * 60 * 60,
                                                np.where(wgts_here == 0, 0, fr0_filt_ip_data[bl][tslice, band]),
                                                wgts=wgts_here,
                                                filter_centers=[0],
                                                filter_half_widths=[FR0_HALFWIDTH / 1000],
                                                mode='dpss_solve',
                                                eigenval_cutoff=[EIGENVAL_CUTOFF],
                                                suppression_factors=[EIGENVAL_CUTOFF],
                                                max_contiguous_edge_flags=len(data.times),
                                                filter_dims=0)
                fr0_filt_ip_data[bl][tslice, band] -= d_mdl

    # save figures to display later
    if not np.all(list(flags.values())):
        waterfall_figs.append(four_pol_inpainting_figure((fr0_filt_ip_data if (FR0_FILTER and not is_auto) else ip_data),
                                                         flags, ip_flags, close=True))
    if covariances:
        covariance_figs.append(covariance_figure(covariances, antpair, data.antpos, close=True))

    # Save inpainting results
    for bl in ip_data:
        if utils.split_bl(bl)[0] == utils.split_bl(bl)[1]:  # is auto
            ip_data[bl][~np.isfinite(ip_data[bl])] = (ip_data[bl][~np.isfinite(ip_data[bl])]).real
    if SAVE_RESULTS:
        assert single_bl_file.endswith(SINGLE_BASELINE_SUFFIX), f'{single_bl_file} does not end with {SINGLE_BASELINE_SUFFIX}.'
        hd.update(data=ip_data, flags=ip_flags, nsamples=ip_nsamples_out)
        hd.history += add_to_history + '\nPrior: ' + prior_note
        outfile = single_bl_file[:-len(SINGLE_BASELINE_SUFFIX)] + INPAINTED_SUFFIX
        print(f"\tNow writing results to {outfile}")
        hd.write_uvh5(outfile, clobber=True)

        # Save fringe-rate filtered results too, if desired (for the auto, just a copy of the inpainted results)
        if FR0_FILTER:
            if not is_auto:
                hd.update(data=fr0_filt_ip_data)
            hd.write_uvh5(outfile.replace('.uvh5', '.FR0_filtered.uvh5'), clobber=True)

        # Save where_inpainted metadata
        hd.update(flags=where_inpainted)
        uvf = UVFlag(hd, mode='flag', copy_flags=True)
        uvf.history += add_to_history
        where_file = single_bl_file[:-len(SINGLE_BASELINE_SUFFIX)] + WHERE_INPAINTED_SUFFIX
        print(f"\tNow writing where-inpainted metadata to {where_file}")
        uvf.write(where_file, clobber=True)

        # Save the 2D fill's covariances, from which its P regenerates exactly (for the noise of the inpainted and stacked data)
        if covariances:
            covariance_file = single_bl_file[:-len(SINGLE_BASELINE_SUFFIX)] + INPAINT_COVARIANCE_SUFFIX
            print(f"\tNow writing the 2D fill's covariances to {covariance_file}")
            write_inpaint_covariance(covariance_file, covariances, antpair, hd.history)

# *Figure 1: Adjustment of the Sky Model to the Night*

For each baseline with a sky model, how the model was adjusted to the night, $\hat m = m\,(1 + G) + R$ (see *Fitting the sky model to
the night*), one row per polarization:
* **The gain:** $|1 + G|$, where 1 is no change, and $\arg(1 + G)$.
* **The additive correction:** $|R|$ in units of the noise $\sigma_N$ each cell has, or would have had where flagged (from the smooth
  autocorrelations, the filled effective `nsamples` and `NOISE_EXCESS`, as the 2D fill assumes), and $\arg(R)$. Noise units keep the
  sky model's nulls from dominating, as they would in $|R| / |m|$.
* **Inside flagged gaps:** $G$ and $R$ are 2D DPSS models over the whole block, so the gaps show what the ridge leaves of each
  correction where the data constrain it poorly (it relaxes toward no adjustment).

Cells where the sky model is flagged, or where nothing was fit, are blank, as is $|R| / \sigma_N$ in integrations flagged across a whole
band, which have no noise level. Autocorrelations, and baselines without a sky model, have no adjustment to show. Every 4th integration and every 2nd channel are shown, still finer than the panels' pixels.

In [ ]:
for adjustment_fig in adjustment_figs:
    display(adjustment_fig)

# *Figure 2: Covariance Learned by the 2D Wiener Fill*

For each baseline, the covariance $C(\Delta t, \Delta\nu)$ the 2D fill used for the residual about its prior (in autocorrelation
units), for the ee and nn polarizations and one per band, and its Fourier transform $\tilde C(f, \tau)$, the thresholded power
spectrum in fringe rate $f$ and delay $\tau$, rebuilt from what the covariance file stores.
* **Top:** the correlation against time lag, $|C(\Delta t, 0)| / C(0, 0)$, and against frequency lag, $|C(0, \Delta\nu)| / C(0, 0)$,
  out to the lag tapers' zeros: how far into a gap the fill carries information.
* **Middle (low band) and bottom (high band):** $\tilde C$ summed over delay against fringe rate, and summed over fringe rate against
  delay, each normalized to its peak, out to the Nyquist fringe rate and delay. Modes below the threshold are zero, so only the dots
  of nonzero power appear.
* **Each column is a Fourier pair:** the sum over delay is the Fourier transform of the time-lag cut at the top, and the sum over
  fringe rate that of the frequency-lag cut.
* **Legend:** $C(0, 0)$, the variance the fill learned (with the noise removed), in units of the mean noise variance.

In [ ]:
for cov_fig in covariance_figs:
    display(cov_fig)

# *Figure 3: 4-Pol Phase and Amplitude Waterfalls Before and After Inpainting*

In [ ]:
for wf_fig in waterfall_figs:
    display(wf_fig)

## Metadata

In [ ]:
for repo in ['hera_cal', 'hera_qm', 'hera_filters', 'hera_notebook_templates', 'pyuvdata', 'numpy']:
    exec(f'from {repo} import __version__')
    print(f'{repo}: {__version__}')

print(f'Finished execution in {(time.time() - tstart) / 60:.2f} minutes.')

In [ ]:
print(f'Finished execution in {(time.time() - tstart) / 60:.2f} minutes.')